In [1]:
from helper_functions import import_flight_data, extract_wind_data
import sys
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [2]:
data_dir = '/Users/rasmus/Desktop/Fysik/AppliedML2026/Final_project_data/data/ML_data/NBI/FDM data'
wind_dir = '/Users/rasmus/Desktop/Fysik/AppliedML2026/Final_project_data/data/ML_data/TWI data'

df = import_flight_data(data_dir=data_dir,limit=1, recursive=True)


In [3]:
COLUMN_MAP = {
    # time
    "Year": "year",
    "QAR YEAR": "year",
    "Date: Year (Derived)": "year",

    "MONTH": "month",
    "QAR MONTH": "month",
    "Date (month)": "month",

    "DAY": "day",
    "QAR DAY": "day",
    "Date (day)": "day",

    "GMT - Hours (BCD)": "hour",
    "QAR HOUR": "hour",
    "UTC Hours": "hour",

    "GMT - Minutes (BCD)": "minute",
    "QAR MINUTE": "minute",
    "UTC Minutes": "minute",

    "GMT Seconds": "second",
    "QAR SECOND": "second",
    "UTC Seconds": "second",

    # altitude
    "Radio Altitude": "radio_altitude",
    "Radio Height 1": "radio_altitude",

    # vertical acceleration
    "Vertical Acceleration": "vert_acc",
    "Normal acceleration": "vert_acc",
}

In [4]:
N_arrivals = 1

arrival_data = import_flight_data(data_dir=data_dir, 
    limit=N_arrivals, add_source_file=True)#, usecols=['Time (secs)','MONTH', 'DAY', 'Year', 'GMT - Hours (BCD)', 'GMT - Minutes (BCD)', 'GMT Seconds', 'Vertical Acceleration', 'Radio Altitude'])

#arrival_data.columns.tolist()

In [5]:
N_arrivals_tot = 3140
N_arrivals = N_arrivals_tot

arrival_data = import_flight_data(data_dir=data_dir,
    limit=N_arrivals, add_source_file=True, usecols=['month', 'day', 'year', 'hour', 'minute', 'second', 'vert_acc', 'radio_altitude'])




In [7]:
from pathlib import Path
from typing import Optional, Union
import numpy as np

import pandas as pd

COLUMN_MAP = {
    # time
    "Year": "year",
    "QAR YEAR": "year",
    "Date: Year (Derived)": "year",

    "MONTH": "month",
    "QAR MONTH": "month",
    "Date (month)": "month",

    "DAY": "day",
    "QAR DAY": "day",
    "Date (day)": "day",

    "GMT - Hours (BCD)": "hour",
    "QAR HOUR": "hour",
    "UTC Hours": "hour",

    "GMT - Minutes (BCD)": "minute",
    "QAR MINUTE": "minute",
    "UTC Minutes": "minute",

    "GMT Seconds": "second",
    "QAR SECOND": "second",
    "UTC Seconds": "second",

    # altitude
    "Radio Altitude": "radio_altitude",
    "Radio Height 1": "radio_altitude",

    # vertical acceleration
    "Vertical Acceleration": "vert_acc",
    "Normal acceleration": "vert_acc",
}

def import_flight_data(
    data_dir: Optional[Union[str, Path]] = None,
    pattern: str = "*.csv",
    recursive: bool = True,
    start: Optional[int] = 0,
    limit: Optional[int] = None,
    add_source_file: bool = True,
    **read_csv_kwargs,
) -> pd.DataFrame:
    """
    Import CSV flight data from the turbulens data folder.

    Args:
        data_dir: Directory to search. Defaults to `<project_root>/data`.
        pattern: File match pattern, default is '*.csv'.
        recursive: If True, search subfolders recursively.
        start: Optional index to start loading files from.
        limit: Optional max number of files to load.
        add_source_file: If True, add a `source_file` column.
        **read_csv_kwargs: Extra keyword arguments passed to `pd.read_csv`.

    Returns:
        A concatenated pandas DataFrame containing all loaded CSV files.
    """
    root_data_dir = Path(__file__).resolve().parents[1] / "data"
    target_dir = Path(data_dir) if data_dir else root_data_dir

    finder = "rglob" if recursive else "glob"

    arrivals_dir = target_dir / "Arrivals"
    departures_dir = target_dir / "Departures"

    arrivals = sorted(getattr(arrivals_dir, finder)(pattern))
    departures = sorted(getattr(departures_dir, finder)(pattern))
    csv_files = arrivals + departures


    if limit is not None:
        csv_files = csv_files[start:start + limit]
    else:
        csv_files = csv_files[start:]

    if not csv_files:
        raise FileNotFoundError(
            f"No files found in '{target_dir}' using pattern '{pattern}'."
        )

    frames = []
    for csv_path in csv_files:
        if 'GKN' in csv_path.name: #Skip files with OY-GKN, as their format sucks
            continue
        frame = pd.read_csv(csv_path, low_memory=False)

        frame.columns = [COLUMN_MAP.get(c, c) for c in frame.columns]       
        needed = [
            'year','month','day',
            'hour','minute','second',
            'vert_acc','radio_altitude']      
          
        frame = frame[[c for c in needed if c in frame.columns]]        
        mask_time = np.isclose(frame['second'] % 1, 0) #!!!
        mask_alt = frame['radio_altitude'] <= 1500 #!!!
        frame = frame.loc[mask_time & mask_alt].reset_index(drop=True) #!!!

        if add_source_file:
            frame["source_file"] = str(csv_path)
        frames.append(frame)

    return pd.concat(frames, ignore_index=True)

def extract_wind_data(flight_df, data_dir, offset=10): 
    """
    Extracts wind data from the TWI dataset for a given flight, based on the landing time and an offset for the recording start time.
    Args:
        flight_df (pd.DataFrame): A DataFrame containing data for one flight, including landing time information.
        data_dir (str): The directory containing the wind data files.
        offset (int): The number of minutes before the landing time to start recording wind data. Default is 10 minutes.
    Returns:
        pd.DataFrame: A DataFrame containing the wind data for the specified time range.
    """

    first = flight_df.iloc[0]

    month = first['month'].astype(int)
    day = first['day'].astype(int)
    if day < 10:
        day_string = '0'+str(day)
    else:
        day_string = str(day)

    if month < 10:
        month_string = '0'+str(month)
    else:        
        month_string = str(month)

    year = first['year'].astype(int)
    start_h = first['hour'].astype(int)
    start_m = first['minute'].astype(int)
    start_s = first['second'].astype(int)
    
    start_of_landing_time = pd.Timestamp(year=year,month=month,day=day,hour=start_h,minute=start_m,second=start_s)

    start_of_recording_time = start_of_landing_time - pd.Timedelta(minutes=offset)
    
    if start_of_recording_time.day != start_of_landing_time.day:
        raise ValueError("Start of recording time is on a different day than the landing time. Please adjust the offset or check the flight data.")
    
    date_string = str(year)+'-'+month_string+'-'+day_string
    file_string = 'TWI-'+date_string+'_UTC_log.csv'

    wind_df = pd.read_csv(data_dir + '/' + file_string, sep=';')

    wind_df['DateTime'] = pd.to_datetime(wind_df['DateTime'], format='%Y-%m-%d %H:%M:%S')

    mask_time = (wind_df['DateTime'] >= start_of_recording_time) & (wind_df['DateTime'] <= start_of_landing_time)

    wind_df = wind_df.loc[mask_time].reset_index(drop=True)
    wind_df['source_file'] = first['source_file']
    return wind_df

In [8]:
full_wind_data = []
idx = 0

for flight_id, flight_df in arrival_data.groupby('source_file'):
    idx +=1
    print(f"Processing flight: {idx}/{len(arrival_data.groupby('source_file'))}")
    try:
        wind_df = extract_wind_data(flight_df, data_dir=wind_dir)
        wind_df['source_file'] = flight_id
        full_wind_data.append(wind_df)

    except ValueError as e:
        print(f"Error processing flight {flight_id}: {e}")
full_wind_data = pd.concat(full_wind_data, ignore_index=True)
full_wind_data.to_csv('/Users/rasmus/Desktop/Fysik/AppliedML2026/AppMLFinalProject.csv', index=False)

Processing flight: 1/2890
Processing flight: 2/2890
Processing flight: 3/2890
Processing flight: 4/2890
Processing flight: 5/2890
Processing flight: 6/2890
Processing flight: 7/2890
Processing flight: 8/2890
Processing flight: 9/2890
Processing flight: 10/2890
Processing flight: 11/2890
Processing flight: 12/2890
Processing flight: 13/2890
Processing flight: 14/2890
Processing flight: 15/2890
Processing flight: 16/2890
Processing flight: 17/2890
Processing flight: 18/2890
Processing flight: 19/2890
Processing flight: 20/2890
Processing flight: 21/2890
Processing flight: 22/2890
Processing flight: 23/2890
Processing flight: 24/2890
Processing flight: 25/2890
Processing flight: 26/2890
Processing flight: 27/2890
Processing flight: 28/2890
Processing flight: 29/2890
Processing flight: 30/2890
Processing flight: 31/2890
Processing flight: 32/2890
Processing flight: 33/2890
Processing flight: 34/2890
Processing flight: 35/2890
Processing flight: 36/2890
Processing flight: 37/2890
Processing

In [9]:
alldata = pd.read_csv('AppMLFinalProject.csv')

In [11]:
alldata.head()

,DateTime,PcTime,ArrRwy04,ArrRwy22,DepRwy22,DepRwy04,ArrRwy04DepRwy22,ArrRwy22DepRwy04,W04.VectorMeanWindSpeed,W04.VectorMeanWindDirection,...,W22.ScalarMaxWindSpeed,W22.ScalarMaxWindAtDirection,W22.ScalarMeanWindSpeed,W22.ScalarWindSpeedDeviation,W22.ScalarMeanWindDirection,W22.ScalarWindDirectionDeviation,W22.LastWindSpeed,W22.LastWindDirection,W22.OkPct,source_file
0,2024-11-28 12:04:29,2024-11-28 10:04:30,2,2,2,2,0,0,3.446184,65.886639,...,9.1,36,7.655263,0.986940,42.496327,6.344822,7.5,38,95.0,/Users/rasmus/Desktop/Fysik/AppliedML2026/Fina...
1,2024-11-28 12:04:35,2024-11-28 10:04:35,2,2,2,2,0,0,3.477918,66.050344,...,9.1,36,7.613158,1.000045,42.311610,6.212914,6.7,45,95.0,/Users/rasmus/Desktop/Fysik/AppliedML2026/Fina...
2,2024-11-28 12:04:38,2024-11-28 10:04:40,2,2,2,2,0,0,3.500163,66.170129,...,9.1,36,7.584211,1.027139,42.152913,6.245114,6.1,38,95.0,/Users/rasmus/Desktop/Fysik/AppliedML2026/Fina...
3,2024-11-28 12:04:44,2024-11-28 10:04:45,2,2,2,2,0,0,3.582905,66.839029,...,9.1,36,7.544737,1.070587,41.626817,5.860641,6.2,41,95.0,/Users/rasmus/Desktop/Fysik/AppliedML2026/Fina...
4,2024-11-28 12:04:50,2024-11-28 10:04:50,2,2,2,2,0,0,3.604055,67.683592,...,9.1,36,7.636842,0.996948,41.207986,5.333008,7.7,44,95.0,/Users/rasmus/Desktop/Fysik/AppliedML2026/Fina...


In [14]:
df_ny = alldata.groupby('source_file').agg(mean_vector_speed=('W04.VectorMeanWindSpeed', 'mean'))

df_ny

,mean_vector_speed
source_file,
/Users/rasmus/Desktop/Fysik/AppliedML2026/Final_project_data/data/ML_data/NBI/FDM data/Arrivals/Nuuk Arrival_ 207724_OY-GRH_28_11_24_16_12.csv,5.553461
/Users/rasmus/Desktop/Fysik/AppliedML2026/Final_project_data/data/ML_data/NBI/FDM data/Arrivals/Nuuk Arrival_ 207725_OY-GRP_28_11_24_43_11.csv,2.643749
/Users/rasmus/Desktop/Fysik/AppliedML2026/Final_project_data/data/ML_data/NBI/FDM data/Arrivals/Nuuk Arrival_ 207731_OY-GRG_28_11_24_41_14.csv,1.846983
/Users/rasmus/Desktop/Fysik/AppliedML2026/Final_project_data/data/ML_data/NBI/FDM data/Arrivals/Nuuk Arrival_ 207732_OY-GRK_28_11_24_31_15.csv,1.224865
/Users/rasmus/Desktop/Fysik/AppliedML2026/Final_project_data/data/ML_data/NBI/FDM data/Arrivals/Nuuk Arrival_ 207734_OY-GRH_28_11_24_56_15.csv,0.634475
...,...
/Users/rasmus/Desktop/Fysik/AppliedML2026/Final_project_data/data/ML_data/NBI/FDM data/Arrivals/Nuuk Dep_ 224651_OY-GRP_18_11_25_06_11.csv,10.904864
/Users/rasmus/Desktop/Fysik/AppliedML2026/Final_project_data/data/ML_data/NBI/FDM data/Arrivals/Nuuk Dep_ 224664_OY-GRK_18_11_25_48_15.csv,0.569633
/Users/rasmus/Desktop/Fysik/AppliedML2026/Final_project_data/data/ML_data/NBI/FDM data/Arrivals/Nuuk Dep_ 224669_OY-GRM_18_11_25_38_16.csv,7.972804
